# Catalog DocIntel Utils Smoke Test

Use this notebook to test catalog-backed Doc Intelligence JSON access before wiring it into local agent logic.

What it covers:
- resolve the catalog Doc Intelligence root
- preview Doc Intelligence JSON paths by docket
- count matching exports when needed
- load one exported JSON payload and inspect key fields

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from review_pipeline_v1.catalog_utils import (
    count_catalog_docintel_exports,
    load_docintel_export,
    parse_docket_filter,
    preview_catalog_docintel_paths,
    resolve_catalog_spark_session,
    resolve_docintel_output_root,
)

In [2]:
DOCINTEL_ROOT = "/Volumes/usdo_aa_catalog/research_tam_datasets/federal_sentencing/cases/docintel_text"
EXECUTION_ENV = "local"
DOCKET_IDS = None  # Example: ["16648584", "16661467"]
PREVIEW_LIMIT = 20
COUNT_MATCHES = False
LIMIT = None
SORT_PATHS = False

docintel_root = resolve_docintel_output_root(
    execution_env="local",
    output_root=DOCINTEL_ROOT,
)
spark = resolve_catalog_spark_session(app_name="review-pipeline-v1-docintel-smoke-test")
docket_filter = parse_docket_filter(DOCKET_IDS)

config_summary = {
    "execution_env": EXECUTION_ENV,
    "docintel_root": str(docintel_root),
    "spark_type": type(spark).__name__,
    "docket_ids": DOCKET_IDS,
    "preview_limit": PREVIEW_LIMIT,
    "count_matches": COUNT_MATCHES,
    "limit": LIMIT,
    "sort_paths": SORT_PATHS,
}
print(json.dumps(config_summary, indent=2))

c:\Users\sonutka\main\2-Envs\tam-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "execution_env": "local",
  "docintel_root": "\\Volumes\\usdo_aa_catalog\\research_tam_datasets\\federal_sentencing\\cases\\docintel_text",
  "spark_type": "SparkSession",
  "docket_ids": null,
  "preview_limit": 20,
  "count_matches": false,
  "limit": null,
  "sort_paths": false
}


In [3]:
preview_rows = preview_catalog_docintel_paths(
    input_root=docintel_root,
    docket_filter=docket_filter,
    preview_limit=PREVIEW_LIMIT,
    sort_paths=SORT_PATHS,
    execution_env=EXECUTION_ENV,
    spark=spark,
)
if not preview_rows:
    raise RuntimeError("No matching Doc Intelligence exports were found for the current filters.")

preview_df = pd.DataFrame(preview_rows)
preview_df

,preview_index,docket_id,docintel_path,source_file_name
0,1,16661467,\Volumes\usdo_aa_catalog\research_tam_datasets...,146737682_sentencing_memo_Sentencing_Memorandu...
1,2,16799958,\Volumes\usdo_aa_catalog\research_tam_datasets...,182087994_sentencing_memo_Sentencing_Memorandu...
2,3,16673142,\Volumes\usdo_aa_catalog\research_tam_datasets...,171265323_sentencing_memo_Sentencing_Memorandu...
3,4,16717269,\Volumes\usdo_aa_catalog\research_tam_datasets...,166886832_sentencing_memo_Sentencing_Memorandu...
4,5,16786587,\Volumes\usdo_aa_catalog\research_tam_datasets...,136164452_fact_Plea_Agreement_Memorandum.docin...
5,6,16833229,\Volumes\usdo_aa_catalog\research_tam_datasets...,174178249_sentencing_memo_Sentencing_Memorandu...
6,7,16858822,\Volumes\usdo_aa_catalog\research_tam_datasets...,125971451_fact_Plea_Agreement.docintel.json
7,8,16804130,\Volumes\usdo_aa_catalog\research_tam_datasets...,193404647_fact_Plea_Agreement.docintel.json
8,9,16717269,\Volumes\usdo_aa_catalog\research_tam_datasets...,120644378_fact_Plea_Agreement.docintel.json
9,10,16784707,\Volumes\usdo_aa_catalog\research_tam_datasets...,377069461_fact_Plea_Agreement.docintel.json


In [4]:
if COUNT_MATCHES:
    total_matches = count_catalog_docintel_exports(
        input_root=docintel_root,
        docket_filter=docket_filter,
        limit=LIMIT,
        sort_paths=SORT_PATHS,
        execution_env=EXECUTION_ENV,
        spark=spark,
    )
    print(json.dumps({"total_matches": total_matches}, indent=2))
else:
    print(json.dumps({"total_matches": "skipped"}, indent=2))

{
  "total_matches": "skipped"
}


In [5]:
sample_docintel_path = Path(preview_rows[0]["docintel_path"])
sample_payload = load_docintel_export(
    sample_docintel_path,
    execution_env=EXECUTION_ENV,
    spark=spark,
)
page_preview = []
for page in sample_payload.get("pages", [])[:2]:
    page_preview.append({
        "page_number": page.get("page_number"),
        "preview": str(page.get("text") or "")[:300],
    })
print(json.dumps({
    "docintel_path": str(sample_docintel_path),
    "docket_id": sample_payload.get("docket_id"),
    "source_pdf_path": sample_payload.get("source_pdf_path"),
    "page_count": sample_payload.get("page_count"),
    "content_length": sample_payload.get("content_length"),
}, indent=2))
pd.DataFrame(page_preview)

{
  "docintel_path": "\\Volumes\\usdo_aa_catalog\\research_tam_datasets\\federal_sentencing\\cases\\docintel_text\\16661467\\146737682_sentencing_memo_Sentencing_Memorandum.docintel.json",
  "docket_id": "16661467",
  "source_pdf_path": "/Volumes/usdo_aa_catalog/research_tam_datasets/federal_sentencing/cases/pdfs/16661467/146737682_sentencing_memo_Sentencing_Memorandum.pdf",
  "page_count": 57,
  "content_length": 117052
}


,page_number,preview
0,1,Case 1:20-cr-00006-KBJ Document 23 Filed 09/24...
1,2,Case 1:20-cr-00006-KBJ Document 23 Filed 09/24...
